# Urban Flow Analytics Datathon 2026

## Merge Conflicts — Integrated Competition Solution

This notebook presents the verified end-to-end solution as one operational story. It reuses completed project artifacts and does not rerun large-scale cleaning, model training, test evaluation, or forecasting.

## 1. Project Overview

The Urban Flow Analytics challenge asks teams to turn large-scale taxi trip records into evidence that improves operational decision-making. Team **Merge Conflicts** built a shared analytical foundation and five connected workstreams:

- pre-trip fare prediction;
- pre-trip trip-duration prediction;
- hourly demand forecasting;
- spatial and origin–destination analysis;
- a business dashboard and safe AI mobility assistant.

The solution links data quality controls to predictive and descriptive outputs, then exposes those verified outputs through decision-support interfaces.

In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "reports").is_dir():
    candidate = PROJECT_ROOT.parent
    if (candidate / "reports").is_dir():
        PROJECT_ROOT = candidate

REQUIRED_ARTIFACTS = [
    "data/feature_contract.md",
    "reports/data_cleaning_decisions.md",
    "docs/MEMBER1_HANDOVER.md",
    "notebooks/01_data_audit_and_cleaning.ipynb",
    "notebooks/02_FarePrediction.ipynb",
    "notebooks/03_DurationPrediction.ipynb",
    "docs/MEMBER2_FARE_HANDOVER.md",
    "docs/MEMBER2_DURATION_HANDOVER.md",
    "models/fare_pipeline.pkl",
    "models/duration_pipeline.pkl",
    "notebooks/03_demand_and_spatial_analysis.ipynb",
    "reports/demand_spatial_findings.md",
    "reports/member3_forecast_metrics.csv",
    "reports/member3_forecasting_summary.json",
    "reports/forecast_24h.csv",
    "reports/forecast_48h.csv",
    "reports/forecast_72h.csv",
    "reports/bonus_track6_summary.md",
    "reports/bonus_track5_summary.md",
    "dashboard/README.md",
    "assistant/README.md",
]
missing_artifacts = [path for path in REQUIRED_ARTIFACTS if not (PROJECT_ROOT / path).is_file()]
assert not missing_artifacts, f"Missing required artifacts: {missing_artifacts}"
print(f"Verified source artifacts: {len(REQUIRED_ARTIFACTS)} / {len(REQUIRED_ARTIFACTS)}")
print("Large Parquet datasets loaded: NO")
print("Model training or test evaluation executed: NO")

Verified source artifacts: 21 / 21
Large Parquet datasets loaded: NO
Model training or test evaluation executed: NO


## 2. Dataset Description

The source data comprises 12 monthly taxi trip files and a zone reference table that maps location identifiers to readable borough, zone, and service-zone fields. Member 1 produced the trusted processed view without modifying the raw files.

| Dataset fact | Verified value |
|---|---:|
| Processed rows | 45,533,334 |
| Processed columns | 45 |
| Parquet row groups | 247 |
| Coverage | 2025-04-01 to 2026-03-31 |

The modelling splits preserve time order and contain no shuffled membership.

In [2]:
split_summary = pd.DataFrame([
    {"Split": "Train", "Rows": 31_988_176, "Pickup period": "Before 2025-12-11"},
    {"Split": "Validation", "Rows": 6_814_901, "Pickup period": "2025-12-11 to 2026-02-04"},
    {"Split": "Test", "Rows": 6_730_257, "Pickup period": "2026-02-05 to 2026-03-31"},
])
assert int(split_summary["Rows"].sum()) == 45_533_334
split_summary

,Split,Rows,Pickup period
0,Train,31988176,Before 2025-12-11
1,Validation,6814901,2025-12-11 to 2026-02-04
2,Test,6730257,2026-02-05 to 2026-03-31


## 3. Data Quality Assessment

Member 1 measured anomalies across all **48,601,782 raw taxi rows** before deciding how each issue should affect task-specific analytical views.

| Anomaly | Count | Percent of raw rows | Interpretation |
|---|---:|---:|---|
| Negative `base_fare` | 2,400,031 | 4.938154% | Signed financial behavior; not proven to be refunds |
| Negative distance | 0 | 0.000000% | No observed cases |
| Zero distance with nonzero fare | 1,471,746 | 3.028173% | Potential waiting, short trips, or incomplete distance readings |
| `rider_count == 0` | 231,578 | 0.476480% | Concentrated by provider; may reflect recording/default behavior |
| Nonpositive duration | 651,610 | 1.340712% | Unusable as recorded for duration and speed modelling |
| Speed above 100 mph | 11,899 | 0.024483% | Extreme trip-average speed used as a conservative model-view exclusion |
| Historical 2008/2009 timestamps | 8 | 0.000016% | Conflict with the stated observation period |
| Either timestamp outside source month | 19,646 | 0.040422% | Mostly legitimate month-boundary trips |

The audit also found mixed overlap between categories, so individual anomaly counts were never added to estimate unique exclusions.

## 4. Cleaning Decisions

Cleaning was implemented as explicit, task-aware eligibility rules while preserving the raw source:

- negative base or final charges were retained for accounting context but excluded from nonnegative fare modelling;
- nonpositive durations and speeds above 100 mph were excluded from time/speed modelling views without rewriting values;
- confirmed 2008/2009 records were quarantined from period analysis, with no invented replacement dates;
- legitimate month-boundary trips, valid zero-distance records, and zero-rider records were retained;
- ambiguous zero-distance and zero-rider cases received audit flags instead of blanket deletion or imputation;
- no semantic imputation or unsupported timestamp correction was applied where field meaning was unclear.

The resulting shared modelling view contains **45,533,334 rows**. Original numeric values and location IDs were retained, and readable zone fields were joined without duplicate expansion.

## 5. Feature Engineering

The shared processed view adds reproducible analytical fields:

| Derived field | Definition and use |
|---|---|
| `trip_duration_minutes` | Dropoff minus pickup time in minutes |
| `speed_mph` | Distance divided by positive duration hours |
| `pickup_date` | Pickup calendar date |
| `pickup_hour` | Pickup hour, 0–23 |
| `day_of_week` | Readable pickup weekday |
| `month` | Pickup month |
| `weekend` | Saturday/Sunday indicator |
| `route_id` | Reproducible origin–destination identifier |

The feature contract excludes dropoff timestamps, targets-as-features, speed-derived leakage, post-trip charges, audit fields, and provenance fields from pre-trip models. **`distance_miles` is valid in the final fare and duration configurations only when it represents an estimated route distance available before departure.**

## 6. Exploratory Analysis

Demand varies materially across the day. The busiest clock hour is **18:00–19:00**, with **3,211,841 retained pickups** over the observation year. Demand is also geographically concentrated around Manhattan and major transport hubs.

![Hourly retained pickup demand](reports/figures/demand_by_hour.png)

The chart summarizes retained trips and does not prove unmet demand, vehicle shortages, or causal trip purpose.

## 7. Fare Prediction

The final fare estimator is `LinearRegression` inside an sklearn pipeline with median numeric imputation, most-frequent categorical imputation, and unknown-safe one-hot encoding.

**Features:** `pickup_hour`, `month`, conditional `distance_miles`, `provider_code`, `day_of_week`, `weekend`, `origin_loc_id`, and `dest_loc_id`.

| Dataset | MAE | RMSE | R² |
|---|---:|---:|---:|
| Validation | 5.737909 | 10.986343 | 0.643793 |
| Final test | 5.571799 | 10.324520 | 0.676153 |

`route_id` was excluded because one-hot encoding produced **22,392 features**. Adding conditional distance reduced validation MAE from 8.689478 to 5.737909, an improvement of approximately **34%**. LinearRegression remained stronger than the tested tree ensembles, and the chronological test was evaluated once only.

![Fare predictions versus actuals](reports/figures/fare_predicted_vs_actual.png)

![Fare MAE by distance bucket](reports/figures/fare_mae_by_distance_bucket.png)

## 8. Trip Duration Prediction

The final duration estimator uses the same eight-input pipeline structure and the same conditional route-distance assumption.

| Dataset | MAE (minutes) | RMSE (minutes) | R² |
|---|---:|---:|---:|
| Validation | 6.367238 | 23.700579 | 0.177008 |
| Final test | 5.802654 | 22.259013 | 0.206474 |

Validation MAE was lowest at **02:00** and highest at **05:00**. The **0–2 mile** bucket performed best, while **20+ miles** performed worst. The maximum validation duration was **8,611.866667 minutes**; **9,158 trips (0.134382%)** exceeded 120 minutes. These observations were retained unchanged, and their long residual tail materially increases RMSE.

![Duration predictions versus actuals](reports/figures/duration_predicted_vs_actual.png)

![Duration MAE by distance bucket](reports/figures/duration_mae_by_distance_bucket.png)

## 9. Demand Forecasting

Member 3 forecast hourly pickup counts for ten high-volume zones. Model choice varies by operational horizon.

| Horizon | Baseline MAE | Improved MAE | Baseline RMSE | Improved RMSE | Winner |
|---:|---:|---:|---:|---:|---|
| 24h | 33.603788911 | 32.302992781 | 53.826564740 | 52.090000550 | Gradient Boosting |
| 48h | 33.574418557 | 34.268282886 | 53.772857929 | 55.024541569 | Seasonal Baseline |
| 72h | 33.571390821 | 35.188656381 | 53.751538591 | 56.369612289 | Seasonal Baseline |

Gradient Boosting supports tactical next-day dispatch at 24 hours. The seasonal baseline is more robust for 48- and 72-hour staffing horizons.

![Demand forecast comparison](reports/figures/demand_forecast.png)

## 10. Spatial / OD Analytics

The three leading pickup zones are **Upper East Side South (2,014,915)**, **JFK Airport (1,923,134)**, and **Midtown Center (1,922,045)**. The largest readable directed zone pair is **Upper East Side South → Upper East Side North**, with **290,297 trips**.

At borough level, **Manhattan → Manhattan** accounts for **36,234,761 trips**, or **79.58%** of retained records. Normalized evening intensity is **2.08×** morning intensity. The hourly demand figure in Section 6 provides the temporal context for this comparison.

![Pickup and destination hotspots](reports/figures/hotspot_zones.png)

![Leading origin-destination flows](reports/figures/od_flows.png)

These are occupied-trip volumes. They do not directly measure empty-vehicle supply, unmet requests, or causal commute behavior.

## 11. Model Evaluation Summary

### Fare validation comparison

| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| Median baseline | 11.713890 | 19.724010 | -0.148117 |
| LinearRegression | 8.689478 | 13.928211 | 0.427485 |
| RandomForest | 9.209341 | 14.128331 | 0.410915 |
| ExtraTrees | 9.106238 | 14.121436 | 0.411490 |
| LinearRegression + conditional distance | 5.737909 | 10.986343 | 0.643793 |

### Duration validation comparison

| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| Median baseline | 9.756819 | 26.446473 | -0.024739 |
| LinearRegression | 8.566630 | 24.971912 | 0.086347 |
| RandomForest | 8.708620 | 25.046847 | 0.080855 |
| ExtraTrees | 8.696595 | 25.090268 | 0.077666 |
| LinearRegression + conditional distance | 6.367238 | 23.700579 | 0.177008 |

### Demand forecast test comparison

| Horizon | Seasonal baseline MAE / RMSE | Gradient Boosting MAE / RMSE | Selected method |
|---:|---:|---:|---|
| 24h | 33.603788911 / 53.826564740 | 32.302992781 / 52.090000550 | Gradient Boosting |
| 48h | 33.574418557 / 53.772857929 | 34.268282886 / 55.024541569 | Seasonal Baseline |
| 72h | 33.571390821 / 53.751538591 | 35.188656381 / 56.369612289 | Seasonal Baseline |

## 12. Key Findings

- Estimated route distance is a strong fare and duration predictor when it is truly available before departure.
- LinearRegression outperformed the tested RandomForest and ExtraTrees configurations on both supervised tasks while training much faster.
- Gradient Boosting improves 24-hour demand forecasts; the seasonal baseline is stronger at 48 and 72 hours.
- Demand is concentrated in Manhattan and major hubs, with Manhattan-to-Manhattan trips representing 79.58% of retained records.
- Evening demand intensity is 2.08 times morning intensity.
- Duration prediction remains harder than fare prediction because long-duration observations and unmeasured operating conditions create substantial residual variation.

## 13. Business Recommendations

The five recommendations below are carried directly from the verified Member 3 findings:

1. Prioritize a staging-capacity review in **Upper East Side South**, the first-ranked pickup zone with **2,014,915** retained pickups (4.43% of all retained trips). Use observed queues and utilization to size any deployment; the counts alone do not establish a vehicle shortage. Evidence: `reports/member3_pickup_zone_totals.csv` and `reports/figures/hotspot_zones.png`.

2. Plan an **evening 16:00–20:00** dispatch emphasis in **Midtown Center**, which records **629,425** pickups in that window (431.11 per nominal hour). Network-wide evening intensity is **8,139.62** pickups/hour versus **3,920.65** in the morning. Evidence: `reports/member3_spatial_pickup_by_time.csv`, `reports/member3_spatial_time_summary.csv`, and `reports/figures/demand_by_hour.png`.

3. Keep **20:00–06:00** dispatch coverage under review for **Upper East Side South → Upper East Side North**, the leading readable late-night pair with **46,364** trips. Late-night network volume is **13,879,471**, but its ten-hour window averages only **3,802.59** pickups/hour; avoid allocating staff from window totals alone. Evidence: `reports/member3_spatial_top_time_pairs.csv` and `reports/figures/member3_od_by_time.png`.

4. Monitor both directions of **Upper East Side South ↔ Upper East Side North** when planning repositioning. The leading direction carries **290,297** trips versus **247,748** in reverse, a difference of **42,549** over the year. Check time-specific vehicle availability before moving empty vehicles: these are occupied-trip counts, not a fleet balance. Evidence: `reports/member3_od_zone_pairs.csv` and `reports/figures/od_flows.png`.

5. Use the **24h gradient-boosting forecast** as the starting point for next-day dispatch and the **seasonal baseline at 48h/72h** for longer-range staffing, subject to ongoing monitoring. Held-out MAE favors those methods at each horizon. This choice is suggested by the observed test results, not an independently validated deployment policy. Evidence: `reports/member3_forecast_metrics.csv` and `reports/figures/demand_forecast.png`.

## 14. Bonus Track Summary

The bonus tracks expose verified outputs without replacing the mandatory analytical work.

### Track 6 — Business Analytics Dashboard

The Streamlit dashboard provides an executive overview, demand and hotspot analysis, OD/time patterns, forecasting comparisons, and evidence-linked recommendations. It runs from a checksum-validated bundle of small reporting outputs and does not require raw trips or model fitting.

### Track 5 — AI Mobility Assistant

The assistant uses deterministic intent routing and safe predefined analytics functions over the verified bundled data. It executes no arbitrary Python and generates no unrestricted SQL. Unsupported or ambiguous questions receive bounded clarification rather than fabricated analysis.

## 15. Solution Architecture

The integrated solution follows this flow:

```text
Raw Taxi Dataset + Zone Reference
                ↓
      Data Validation / Cleaning
                ↓
       Shared Feature Engineering
                ↓
Fare Model | Duration Model | Demand Forecast
                ↓
        Spatial / OD Analytics
                ↓
      Verified Analytics Outputs
                ↓
Business Dashboard | AI Mobility Assistant
                ↓
     Operational Decision Support
```

![End-to-end Urban Flow Analytics solution architecture](reports/figures/MergeConflicts_Architecture_Diagram.png)

The architecture links confidential raw taxi and zone-reference inputs to validation, cleaning, and shared leakage-safe features. Those features support fare and duration prediction plus horizon-specific demand forecasting, followed by spatial and OD analysis and a verified reporting layer. The dashboard and deterministic assistant consume those verified outputs to support fleet positioning, staffing, planning, and demand-aware operations.

## 16. Limitations

- `distance_miles` requires a valid pre-trip route estimate; completed-trip mileage is not a deployable pre-trip input.
- Duration has a heavy long-duration tail that strongly affects RMSE.
- Traffic, congestion, weather, incidents, and road conditions are not fully represented.
- Fare and duration final fitting used deterministic train-plus-validation samples rather than all approximately 39 million available rows.
- One global missing demand hour, 2026-03-08 02:00, remains documented as zero; no timezone correction was inferred.
- Forecast performance differs materially by horizon, and the ten-zone cohort was selected retrospectively.
- Competition-data confidentiality prevents public sharing of raw, processed, and split datasets.
- Observed trip volumes support planning review but do not independently prove shortages, causal behavior, or deployment impact.

## 17. Conclusion

The project converts a year of taxi and zone-reference data into a governed analytical foundation, pre-trip fare and duration estimates, horizon-specific demand forecasts, and spatial evidence about demand concentration and movement. The dashboard and AI assistant make these verified outputs accessible to operational users while preserving the limits of the evidence. Together, the components support dispatch, staffing, and monitoring decisions without claiming unmeasured financial or causal outcomes.

---

### Reproducibility note

This integrated notebook reads only small verified project artifacts and references committed figures using relative paths. Detailed computation remains in the member notebooks and handover reports. Final fare and duration test evaluations are not rerun here.